# Notebook 2 — Heat Load Analysis

Parasitic heat loads are the primary engineering challenge in DR systems.
This notebook analyzes the two dominant mechanisms:

1. **Wire conduction** — heat conducted along signal/control wiring
2. **Thermal radiation** — radiation from warm shields leaking into colder stages

### Wire conduction
$$P_{\text{wire}} = \frac{n_w \kappa A}{L} \Delta T$$

where $\kappa$ is the mean thermal conductivity of the wire material, $A$ is the cross-section,
$L$ is the wire segment length, and $n_w$ is the number of wires.

### Radiation (Stefan–Boltzmann)
$$P_{\text{rad}} = \varepsilon \sigma A (T_h^4 - T_c^4)$$

where $\varepsilon$ is the emissivity of the radiation shield.

In [ ]:
import sys
sys.path.insert(0, '../python')

import numpy as np
import matplotlib.pyplot as plt
from cryo_thermal import heat_loads, STEFAN_BOLTZMANN

# Default wire parameters (stainless-steel loom, Oxford Triton cabling)
N_WIRES   = 24
WIRE_A    = 2e-8    # m²  (160 µm diameter)
WIRE_L    = 0.3     # m   (distance between stages)
RAD_AREA  = 0.01    # m²  (radiation shield area facing MXC)
EMISS     = 0.05    # aluminium-coated copper shield

print(f'Wire parameters: {N_WIRES} wires, A = {WIRE_A*1e6:.1f} mm², L = {WIRE_L*100:.0f} cm')
print(f'Radiation: area = {RAD_AREA*1e4:.0f} cm², ε = {EMISS}')

# Compute loads for each stage pair
stage_pairs = [
    ('300K → 50K',  300.0,   44.0),
    ('50K → 4K',     44.0,    3.7),
    ('4K → Still',    3.7,  0.476),
    ('Still → CP',  0.476,  0.033),
    ('CP → MXC',    0.033, 0.013),
]

print(f"\n{'Stage pair':<18} {'P_wire (µW)':>14} {'P_rad (µW)':>12} {'Total (µW)':>12}")
print('-' * 60)
for label, T_h, T_c in stage_pairs:
    P_w, P_r, _ = heat_loads(T_h, T_c, N_WIRES, WIRE_A, WIRE_L, RAD_AREA, EMISS)
    tot = P_w + P_r
    print(f"{label:<18} {P_w*1e6:>14.3f} {P_r*1e6:>12.3f} {tot*1e6:>12.3f}")

In [ ]:
# ── Wire count sensitivity ────────────────────────────────────────────────────
wire_counts = np.arange(8, 100, 4)
# Heat load from 300K to 50K stage
P_wire_300_50 = np.array([
    heat_loads(300.0, 44.0, int(n), WIRE_A, WIRE_L, RAD_AREA, EMISS)[0]
    for n in wire_counts
])
# MXC: heat load from Still to MXC (small wires, 12 lines)
P_wire_mxc = np.array([
    heat_loads(0.476, 0.013, int(n), 5e-9, 0.15, 1e-4, 0.02)[0]
    for n in wire_counts
])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(wire_counts, P_wire_300_50 * 1e-3, 'r-o', ms=4, lw=2)
axes[0].set_xlabel('Number of wires', fontsize=12)
axes[0].set_ylabel('Wire conduction load (mW)', fontsize=12)
axes[0].set_title('300K→50K Stage: Wire Conduction vs. Wire Count', fontsize=11)
axes[0].axvline(24, color='k', ls='--', alpha=0.6, label='Default (24 wires)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(wire_counts, P_wire_mxc * 1e9, 'b-o', ms=4, lw=2)
axes[1].set_xlabel('Number of wires', fontsize=12)
axes[1].set_ylabel('Wire conduction load (nW)', fontsize=12)
axes[1].set_title('Still→MXC: Wire Conduction vs. Wire Count', fontsize=11)
axes[1].axvline(12, color='k', ls='--', alpha=0.6, label='Default (12 wires)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/heat_load_wires.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Emissivity sensitivity ────────────────────────────────────────────────────
emiss_vals = np.linspace(0.01, 0.3, 100)

P_rad_300_50 = np.array([
    heat_loads(300.0, 44.0, N_WIRES, WIRE_A, WIRE_L, RAD_AREA, e)[1]
    for e in emiss_vals
])

# Compare wire vs radiation at 300K→50K
P_wire_default = heat_loads(300.0, 44.0, N_WIRES, WIRE_A, WIRE_L, RAD_AREA, EMISS)[0]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(emiss_vals, P_rad_300_50 * 1e3, 'r-', lw=2.5, label='Radiation (300K→50K)')
ax.axhline(P_wire_default * 1e3, color='b', ls='--', lw=2,
           label=f'Wire conduction (24 wires, fixed) = {P_wire_default*1e3:.1f} mW')
ax.axvline(EMISS, color='k', ls=':', alpha=0.7, label=f'Default ε = {EMISS}')
ax.set_xlabel('Emissivity ε', fontsize=12)
ax.set_ylabel('Heat load (mW)', fontsize=12)
ax.set_title('Radiation vs. Wire Conduction Heat Load (300K → 50K)', fontsize=12)
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/heat_load_emissivity.png', dpi=150, bbox_inches='tight')
plt.show()

P_rad_default = heat_loads(300.0, 44.0, N_WIRES, WIRE_A, WIRE_L, RAD_AREA, EMISS)[1]
print(f'At ε={EMISS}: P_wire = {P_wire_default*1e3:.2f} mW,  P_rad = {P_rad_default*1e3:.3f} mW')
print(f'Radiation is {P_rad_default/P_wire_default*100:.1f}% of wire conduction at 300K→50K')

In [ ]:
# ── MXC budget breakdown ──────────────────────────────────────────────────────
# How much margin does the MXC have against various qubit power levels?
from cryo_thermal import simulate_cooldown, STAGE_NAMES

qubit_powers = [0, 1e-7, 5e-7, 1e-6, 5e-6, 1e-5, 5e-5]  # W
T_mxc_list   = []

for P_q in qubit_powers:
    t, T = simulate_cooldown(t_hours=10.0, params={'P_qubit_W': P_q}, n_eval=1000)
    T_mxc_list.append(T[-1, 5])  # index 5 = MXC

T_mxc_arr = np.array(T_mxc_list) * 1e3  # → mK

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(np.array(qubit_powers) * 1e6, T_mxc_arr, 'ko-', ms=7, lw=2)
ax.axhline(20, color='r', ls='--', lw=1.5, label='20 mK limit (qubit dephasing)')
ax.set_xlabel('Qubit power dissipation (µW)', fontsize=12)
ax.set_ylabel('MXC steady-state temperature (mK)', fontsize=12)
ax.set_title('MXC Temperature vs. Qubit Power Budget', fontsize=12)
ax.legend(fontsize=11); ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../outputs/mxc_qubit_budget.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n{"Qubit power":>14}  T_MXC')
for P_q, T_m in zip(qubit_powers, T_mxc_arr):
    flag = '  ← exceeds 20 mK limit' if T_m > 20 else ''
    print(f'{P_q*1e6:>10.1f} µW  {T_m:.2f} mK{flag}')